## Industry rule for this stage

**Rule:** WOE/IV maps and the scaler/encoder must be `fit` on train only and `applied` (not re-fit) on test. **Naming bug fixed here:** this notebook used to re-save its fully-transformed output as `train_df.csv`/`test_df.csv` — the exact same filenames `02_data_preprocessing` uses for the early, pre-transformation split. That collision meant `drift_detection.ipynb`'s baseline load of "train_df.csv" could silently pick up either the raw cleaned split or the fully scaled/WOE-transformed data, depending only on which notebook ran last. Renamed the final output here to `train_df_final_scaled.csv` / `test_df_final_scaled.csv` so every filename on disk means exactly one thing.


In [756]:
# NOTE (dependency not included in this zip): `ml_project` is a custom local package
# providing helper functions used below (e.g. fill_with_group_median, detect_outliers_and_capping,
# convert_binary_to_bool, generate_binning_column, apply_binning_test). Its source was not part of
# the uploaded files, so this cell will fail until that package is placed on the Python path.
# Industry rule: shared helper functions like these belong in a versioned, importable package/module
# (or a src/ folder), never only on one person's local machine path.
from ml_project import *  # see note above
import sys

from ml_project import *

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Check Point 4 - Feature Transfomration

In [4]:
train_df = pd.read_csv("../data/processed/transformed_train_df.csv")
test_df = pd.read_csv("../data/processed/transformed_test_df.csv")

#### Numerical columns Bininng

In [5]:
copy_train_df = train_df.copy()

In [6]:
import numpy as np

bin_dict = {}

def generate_binning_column(cols_name):

    for col in cols_name:

        if col == "Age":
            copy_train_df['Age_Group'] = pd.cut(
                copy_train_df['Age'],
                bins=[0, 18, 25, 35, 45, 60, 100],
                labels=['Very_Young', 'Young_Adult', 'Early_Career', 'Mid_Career', 'Senior', 'Retired'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['Age'] = [0, 18, 25, 35, 45, 60, 100]

        elif col == 'Income':
            bins = copy_train_df['Income'].quantile([0, 0.33, 0.66, 1.0]).tolist()
            bins[0] = -np.inf
            bins[-1] = np.inf
            copy_train_df['Income_Group'] = pd.cut(
                copy_train_df['Income'],
                bins=bins,
                labels=['Low', 'Medium', 'High'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['Income'] = bins

        elif col == 'Employment_length':
            bins = copy_train_df['Employment_length'].quantile([0, 0.25, 0.50, 0.75, 1.0]).tolist()
            bins[0] = -np.inf
            bins[-1] = np.inf
            copy_train_df['Employment_Length_Group'] = pd.cut(
                copy_train_df['Employment_length'],
                bins=bins,
                labels=['Fresher', 'Mid-Level', 'Senior', 'Very Senior'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['Employment_length'] = bins

        elif col == 'Loan_amount':
            bins = copy_train_df['Loan_amount'].quantile([0, 0.33, 0.66, 1.0]).tolist()
            bins[0] = -np.inf
            bins[-1] = np.inf
            copy_train_df['Loan_Amount_Group'] = pd.cut(
                copy_train_df['Loan_amount'],
                bins=bins,
                labels=['Low', 'Medium', 'High'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['Loan_amount'] = bins

        elif col == 'Interest_rate':
            bins = copy_train_df['Interest_rate'].quantile([0, 0.33, 0.66, 1.0]).tolist()
            bins[0] = -np.inf
            bins[-1] = np.inf
            copy_train_df['Interest_Rate_Group'] = pd.cut(
                copy_train_df['Interest_rate'],
                bins=bins,
                labels=['Low', 'Medium', 'High'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['Interest_rate'] = bins

        elif col == 'Loan_percent_income':
            bins = copy_train_df['Loan_percent_income'].quantile([0, 0.25, 0.50, 0.75, 1.0]).tolist()
            bins[0] = -np.inf
            bins[-1] = np.inf

            copy_train_df['Loan_Percent_Income_Group'] = pd.cut(
                copy_train_df['Loan_percent_income'],
                bins=bins,
                labels=['Low', 'Medium', 'High', 'Very High'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['Loan_percent_income'] = bins

        elif col == 'Credit_history_length':
            bins = copy_train_df['Credit_history_length'].quantile([0, 0.33, 0.66, 1.0]).tolist()
            bins[0] = -np.inf
            bins[-1] = np.inf
            copy_train_df['Credit_History_Group'] = pd.cut(
                copy_train_df['Credit_history_length'],
                bins=bins,
                labels=['New', 'Fair', 'Good'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['Credit_history_length'] = bins

        elif col == 'age_employment_ratio':
            bins = copy_train_df['age_employment_ratio'].quantile([0, 0.33, 0.66, 1.0]).tolist()
            bins[0] = -np.inf
            bins[-1] = np.inf
            copy_train_df['age_employment_ratio_Group'] = pd.cut(
                copy_train_df['age_employment_ratio'],
                bins=bins,
                labels=['Unstable_Career', 'Moderate_Career', 'Stable_Career'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['age_employment_ratio'] = bins

        elif col == 'income_per_credit_year':
            bins = copy_train_df['income_per_credit_year'].quantile([0, 0.33, 0.66, 1.0]).tolist()
            bins[0] = -np.inf
            bins[-1] = np.inf
            copy_train_df['income_per_credit_year_Group'] = pd.cut(
                copy_train_df['income_per_credit_year'],
                bins=bins,
                labels=['Low_Financial_Growth', 'Moderate_Financial_Growth', 'High_Financial_Growth'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['income_per_credit_year'] = bins

        else:
            print(f"{col} not found in dataset")

In [7]:
generate_binning_column(copy_train_df.columns)

Age binned successfully
Income binned successfully
Home_ownership not found in dataset
Employment_length binned successfully
Loan_intent not found in dataset
internal_credit_rating not found in dataset
Loan_amount binned successfully
Interest_rate binned successfully
Loan_percent_income binned successfully
Default not found in dataset
Credit_history_length binned successfully
Loan_status not found in dataset
is_eligible not found in dataset
age_employment_ratio binned successfully
income_per_credit_year binned successfully


#### Calculate WOE and IV value for checking feature strength

In [8]:
copy_train_df.groupby('Interest_Rate_Group')['Loan_status'].value_counts().unstack().fillna(0)

C:\Users\dell\AppData\Local\Temp\ipykernel_868\3885956550.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  copy_train_df.groupby('Interest_Rate_Group')['Loan_status'].value_counts().unstack().fillna(0)


Loan_status,False,True
Interest_Rate_Group,,
Low,7881,881
Medium,6925,1500
High,5455,3290


In [9]:
import pandas as pd
import numpy as np

def calculate_woe_iv(df, cat_col, target_col):

    # Only categorical check
    if not pd.api.types.is_object_dtype(df[cat_col]) and not pd.api.types.is_categorical_dtype(df[cat_col]):
        raise ValueError("Column must be categorical")

    # Create table
    woe_iv_table = df.groupby(cat_col)[target_col].value_counts().unstack().fillna(0)

    # Rename (assuming 0=Good, 1=Bad)
    woe_iv_table.columns = woe_iv_table.columns.map({False: 'Good', True: 'Bad'})

    total_good = (df[target_col] == False).sum()
    total_bad  = (df[target_col] == True).sum() 

    # Distribution
    woe_iv_table['Good_dist'] = woe_iv_table['Good'] / total_good
    woe_iv_table['Bad_dist'] = woe_iv_table['Bad'] / total_bad

    # Avoid division by zero
    woe_iv_table['Good_dist'] = woe_iv_table['Good_dist'].replace(0, 1e-10)
    woe_iv_table['Bad_dist'] = woe_iv_table['Bad_dist'].replace(0, 1e-10)

    # WOE & IV
    woe_iv_table['WOE'] = np.log(woe_iv_table['Good_dist'] / woe_iv_table['Bad_dist'])
    woe_iv_table['IV'] = (woe_iv_table['Good_dist'] - woe_iv_table['Bad_dist']) * woe_iv_table['WOE']

    iv_value = woe_iv_table['IV'].sum()

    return woe_iv_table, iv_value, total_good, total_bad

In [10]:
# Store IV values and tables
iv_results = {}
woe_maps = {}
woe_features = []

binned_cols = ['Age_Group','Income_Group', 'Home_ownership', 'Employment_Length_Group', 'Loan_intent',
                'internal_credit_rating', 'Loan_Amount_Group', 'Interest_Rate_Group',
                'Loan_Percent_Income_Group', 'Credit_History_Group', 'age_employment_ratio_Group', 'income_per_credit_year_Group']

for col in binned_cols:
    woe_iv_table, iv_value, total_good, total_bad = calculate_woe_iv(copy_train_df, col, 'Loan_status')

    iv_results[col] = {'iv_value': iv_value, 'woe_table': woe_iv_table}
    
    # Mapping the Woe values on df1's features
    woe_map = woe_iv_table['WOE'].to_dict()
    woe_maps[col] = woe_map

    copy_train_df[col + "_woe"] = copy_train_df[col].map(woe_map)
    woe_features.append(col + "_woe")

        # print(f"\n{col} - IV: {iv_value:.4f}")
        # print(woe_iv_table)

C:\Users\dell\AppData\Local\Temp\ipykernel_868\3577653312.py:7: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if not pd.api.types.is_object_dtype(df[cat_col]) and not pd.api.types.is_categorical_dtype(df[cat_col]):
C:\Users\dell\AppData\Local\Temp\ipykernel_868\3577653312.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  woe_iv_table = df.groupby(cat_col)[target_col].value_counts().unstack().fillna(0)
C:\Users\dell\AppData\Local\Temp\ipykernel_868\3577653312.py:7: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if not pd.api.types.is_object_dtype(df[cat_col]) and not pd.api.types.is_categorical

In [11]:
# IV interpretation function
def check_iv_strength(iv_value, feature_name):
    print(iv_value)
    if iv_value < 0.02:
        print(f"{feature_name}: Weak Predictor - DELETE ")
    elif 0.02 <= iv_value < 0.1:
        print(f"{feature_name}: Medium Predictor - KEEP ")
    else:
        print(f"{feature_name}: Strong Predictor - KEEP ")

In [12]:
check_iv_strength(iv_results['Age_Group']['iv_value'], 'Age')
check_iv_strength(iv_results['Income_Group']['iv_value'], 'Incomee')
check_iv_strength(iv_results['Home_ownership']['iv_value'], 'Home_ownership')
check_iv_strength(iv_results['Employment_Length_Group']['iv_value'], 'Employment_Length_Group')
check_iv_strength(iv_results['Loan_intent']['iv_value'], 'Loan_intent')
check_iv_strength(iv_results['internal_credit_rating']['iv_value'], 'internal_credit_rating')
check_iv_strength(iv_results['Loan_Amount_Group']['iv_value'], 'Loan_amount')
check_iv_strength(iv_results['Interest_Rate_Group']['iv_value'], 'Interest_Rate_Group')
check_iv_strength(iv_results['Loan_Percent_Income_Group']['iv_value'], 'Loan_Percent_Income_Group')
check_iv_strength(iv_results['Credit_History_Group']['iv_value'], 'Credit_History_Group')
check_iv_strength(iv_results['age_employment_ratio_Group']['iv_value'], 'age_employment_ratio')
check_iv_strength(iv_results['income_per_credit_year_Group']['iv_value'], 'income_per_credit_year')

0.0056244917013092985
Age: Weak Predictor - DELETE 
0.386764270923431
Incomee: Strong Predictor - KEEP 
0.3835214488349805
Home_ownership: Strong Predictor - KEEP 
0.05716817253007478
Employment_Length_Group: Medium Predictor - KEEP 
0.09256182872854508
Loan_intent: Medium Predictor - KEEP 
0.8734166597608912
internal_credit_rating: Strong Predictor - KEEP 
0.05289824821183857
Loan_amount: Medium Predictor - KEEP 
0.47291482985260647
Interest_Rate_Group: Strong Predictor - KEEP 
0.6338395614524255
Loan_Percent_Income_Group: Strong Predictor - KEEP 
0.0033200263300483613
Credit_History_Group: Weak Predictor - DELETE 
0.04415506691750644
age_employment_ratio: Medium Predictor - KEEP 
0.14197059900211537
income_per_credit_year: Strong Predictor - KEEP 


In [13]:
copy_train_df.dtypes

Age                                  float64
Income                               float64
Home_ownership                        object
Employment_length                    float64
Loan_intent                           object
internal_credit_rating                object
Loan_amount                          float64
Interest_rate                        float64
Loan_percent_income                  float64
Default                               object
Credit_history_length                float64
Loan_status                             bool
is_eligible                             bool
age_employment_ratio                 float64
income_per_credit_year               float64
Age_Group                           category
Income_Group                        category
Employment_Length_Group             category
Loan_Amount_Group                   category
Interest_Rate_Group                 category
Loan_Percent_Income_Group           category
Credit_History_Group                category
age_employ

In [14]:
cols = [col for col in copy_train_df.columns if col.endswith('_woe')]

copy_train_df[cols] = copy_train_df[cols].astype(float)

#### Apply Same Mapping (Test)

In [15]:
copy_test_df = test_df.copy()

In [16]:
def apply_binning_test(df):

    df['Age_Group'] = pd.cut(
        df['Age'],
        bins=bin_dict['Age'],
        labels=['Very_Young', 'Young_Adult', 'Early_Career', 'Mid_Career', 'Senior', 'Retired'],
        include_lowest=True
    )

    df['Income_Group'] = pd.cut(
        df['Income'],
        bins=bin_dict['Income'],
        labels=['Low', 'Medium', 'High'],
        include_lowest=True
    )

    df['Employment_Length_Group'] = pd.cut(
        df['Employment_length'],
        bins=bin_dict['Employment_length'],
        labels=['Fresher', 'Mid-Level', 'Senior', 'Very Senior'],
        include_lowest=True
    )

    df['Loan_Amount_Group'] = pd.cut(
        df['Loan_amount'],
        bins=bin_dict['Loan_amount'],
        labels=['Low', 'Medium', 'High'],
        include_lowest=True
    )

    df['Interest_Rate_Group'] = pd.cut(
        df['Interest_rate'],
        bins=bin_dict['Interest_rate'],
        labels=['Low', 'Medium', 'High'],
        include_lowest=True
    )

    df['Loan_Percent_Income_Group'] = pd.cut(
        df['Loan_percent_income'],
        bins=bin_dict['Loan_percent_income'],
        labels=['Low', 'Medium', 'High', 'Very High'],
        include_lowest=True
    )

    df['Credit_History_Group'] = pd.cut(
        df['Credit_history_length'],
        bins=bin_dict['Credit_history_length'],
        labels=['New', 'Fair', 'Good'],
        include_lowest=True
    )

    df['age_employment_ratio_Group'] = pd.cut(
        df['age_employment_ratio'],
        bins=bin_dict['age_employment_ratio'],
        labels=['Unstable_Career', 'Moderate_Career', 'Stable_Career'],
        include_lowest=True
    )

    df['income_per_credit_year_Group'] = pd.cut(
        df['income_per_credit_year'],
        bins=bin_dict['income_per_credit_year'],
        labels=['Low_Financial_Growth', 'Moderate_Financial_Growth', 'High_Financial_Growth'],
        include_lowest=True
    )

    return df

In [17]:
apply_binning_test(copy_test_df)

,Age,Income,Home_ownership,Employment_length,Loan_intent,internal_credit_rating,Loan_amount,Interest_rate,Loan_percent_income,Default,...,income_per_credit_year,Age_Group,Income_Group,Employment_Length_Group,Loan_Amount_Group,Interest_Rate_Group,Loan_Percent_Income_Group,Credit_History_Group,age_employment_ratio_Group,income_per_credit_year_Group
0,28.0,56461.0,Rent,5.0,Education,E,5400.0,15.68,0.10,Y,...,9410.166667,Early_Career,Medium,Senior,Low,High,Medium,Fair,Moderate_Career,Moderate_Financial_Growth
1,30.0,108000.0,Mortgage,0.0,Debtconsolidation,B,4925.0,12.26,0.05,N,...,12000.000000,Early_Career,High,Fresher,Low,Medium,Low,Good,Unstable_Career,Moderate_Financial_Growth
2,29.0,102000.0,Own,10.0,Medical,B,18000.0,9.33,0.18,N,...,14571.428571,Early_Career,High,Very Senior,High,Low,High,Good,Stable_Career,Moderate_Financial_Growth
3,29.0,45000.0,Rent,4.0,Venture,B,10000.0,12.42,0.22,N,...,6428.571429,Early_Career,Medium,Mid-Level,Medium,Medium,High,Good,Moderate_Career,Low_Financial_Growth
4,31.0,30000.0,Rent,4.0,Debtconsolidation,B,12000.0,10.39,0.40,N,...,3333.333333,Early_Career,Low,Mid-Level,High,Medium,Very High,Good,Moderate_Career,Low_Financial_Growth
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6479,23.0,42000.0,Rent,2.0,Education,A,7000.0,7.88,0.17,N,...,14000.000000,Young_Adult,Low,Fresher,Medium,Low,High,New,Unstable_Career,Moderate_Financial_Growth
6480,23.0,62000.0,Mortgage,8.0,Homeimprovement,C,9000.0,11.03,0.15,N,...,31000.000000,Young_Adult,Medium,Very Senior,Medium,Medium,Medium,New,Stable_Career,High_Financial_Growth
6481,41.0,98004.0,Mortgage,25.0,Debtconsolidation,C,12000.0,13.57,0.12,N,...,6533.600000,Mid_Career,High,Very Senior,High,High,Medium,Good,Stable_Career,Low_Financial_Growth
6482,24.0,50000.0,Mortgage,7.0,Venture,D,1000.0,14.42,0.02,N,...,16666.666667,Young_Adult,Medium,Senior,Low,High,Low,New,Stable_Career,Moderate_Financial_Growth


Applying the rules of copy_train_df woe maps to the copy_test_df

In [18]:
for col in woe_maps:
    copy_test_df[col + '_woe'] = copy_test_df[col].map(woe_maps[col])

In [19]:
copy_test_df.columns

Index(['Age', 'Income', 'Home_ownership', 'Employment_length', 'Loan_intent',
       'internal_credit_rating', 'Loan_amount', 'Interest_rate',
       'Loan_percent_income', 'Default', 'Credit_history_length',
       'Loan_status', 'is_eligible', 'age_employment_ratio',
       'income_per_credit_year', 'Age_Group', 'Income_Group',
       'Employment_Length_Group', 'Loan_Amount_Group', 'Interest_Rate_Group',
       'Loan_Percent_Income_Group', 'Credit_History_Group',
       'age_employment_ratio_Group', 'income_per_credit_year_Group',
       'Age_Group_woe', 'Income_Group_woe', 'Home_ownership_woe',
       'Employment_Length_Group_woe', 'Loan_intent_woe',
       'internal_credit_rating_woe', 'Loan_Amount_Group_woe',
       'Interest_Rate_Group_woe', 'Loan_Percent_Income_Group_woe',
       'Credit_History_Group_woe', 'age_employment_ratio_Group_woe',
       'income_per_credit_year_Group_woe'],
      dtype='object')

#### Calculating Final Credit Score

Train Model

In [20]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(copy_train_df[woe_features], copy_train_df['Loan_status'])

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


Predict Probability

In [21]:
copy_train_df['prob'] = model.predict_proba(copy_train_df[woe_features])[:, 1]
copy_test_df['prob'] = model.predict_proba(copy_test_df[woe_features])[:, 1]

Convert To Credit Score (300–900)

In [ ]:
PDO = 50
base_score = 600
base_odds = total_good / total_bad

factor = PDO / np.log(2)
offset = base_score - factor * np.log(base_odds)

# Score formula
copy_train_df['credit_score'] = offset - factor * np.log(copy_train_df['prob'] / (1 - copy_train_df['prob']))
copy_test_df['credit_score'] = offset - factor * np.log(copy_test_df['prob'] / (1 - copy_test_df['prob']))

# Clip to 300–900
copy_train_df['credit_score'] = copy_train_df['credit_score'].clip(300, 900)
copy_test_df['credit_score'] = copy_test_df['credit_score'].clip(300, 900)

#### Basic data anlysis insights on final credit score

Filter customers with low credit score and default status

In [23]:
# filter customers with low credit score
(copy_train_df.loc[
    (copy_train_df['credit_score'] < 650),
    ['credit_score', 'Income', 'Loan_status', 'internal_credit_rating', 'Loan_intent', 'Home_ownership', 'Interest_rate']
]).sort_values(by='internal_credit_rating', ascending=False)

,credit_score,Income,Loan_status,internal_credit_rating,Loan_intent,Home_ownership,Interest_rate
18999,300.000000,46000.0,True,G,Education,Mortgage,19.160
4466,300.000000,49000.0,True,G,Personal,Mortgage,19.820
8149,311.321109,118000.0,True,G,Personal,Mortgage,20.110
22929,300.000000,58650.0,True,G,Medical,Mortgage,20.170
1617,300.000000,54000.0,True,G,Personal,Rent,20.110
...,...,...,...,...,...,...,...
16467,600.962413,30996.0,False,A,Medical,Mortgage,7.660
10729,513.964592,36000.0,False,A,Medical,Rent,5.790
21852,589.673595,30000.0,False,A,Medical,Rent,7.740
7374,607.187144,6000.0,True,A,Personal,Rent,12.672


Filter customers with low credit score but no default

In [24]:
# filter customers with high credit score
copy_train_df.loc[
    (copy_train_df['credit_score'] > 650),
    ['credit_score', 'Income', 'Loan_status', 'Loan_intent', 'Home_ownership', 'Interest_rate']
]

,credit_score,Income,Loan_status,Loan_intent,Home_ownership,Interest_rate
0,756.011963,75800.0,False,Personal,Rent,6.54
2,705.919300,53088.0,False,Personal,Rent,6.54
3,827.799979,225000.0,False,Homeimprovement,Mortgage,7.14
5,805.473709,90000.0,False,Personal,Mortgage,11.49
6,712.633160,54036.0,False,Medical,Mortgage,11.36
...,...,...,...,...,...,...
25926,756.491371,54000.0,False,Personal,Mortgage,5.42
25927,662.141900,25000.0,False,Medical,Mortgage,9.32
25928,711.949086,21600.0,False,Education,Mortgage,5.42
25929,809.004126,81000.0,False,Venture,Rent,7.90


In [25]:
print("\nDefault when credit score is above 650")
print(copy_train_df[copy_train_df['credit_score'] > 650].groupby(['Default']).size())

print("\nDefault when credit score is below 650")
print(copy_train_df[copy_train_df['credit_score'] < 650].groupby(['Default']).size())


print("\nLoan status when credit score is above 650")
print(copy_train_df[copy_train_df['credit_score'] > 650].groupby(['Loan_status']).size())

print("\nLoan status when credit score is below 650")
print(copy_train_df[copy_train_df['credit_score'] < 650].groupby(['Loan_status']).size())


Default when credit score is above 650
Default
N    12312
Y     1186
dtype: int64

Default when credit score is below 650
Default
N    9039
Y    3395
dtype: int64

Loan status when credit score is above 650
Loan_status
False    12792
True       706
dtype: int64

Loan status when credit score is below 650
Loan_status
False    7469
True     4965
dtype: int64


In [26]:
demo_df = copy_train_df[['credit_score', 'Default', 'Loan_status', 'internal_credit_rating']][
    (copy_train_df['credit_score'] > 650) &
    (copy_train_df['Default'] == 'N') &
    (copy_train_df['Loan_status'] == False)
]

print(demo_df['internal_credit_rating'].value_counts())

internal_credit_rating
A    6484
B    4212
C     982
D      39
E       4
Name: count, dtype: int64


In [27]:
demo_df = copy_train_df[['credit_score', 'Default', 'Loan_status', 'internal_credit_rating']][
    (copy_train_df['credit_score'] < 700) &
    (copy_train_df['Default'] == 'N') &
    (copy_train_df['Loan_status'] == False)
]

print(demo_df['internal_credit_rating'].value_counts())

internal_credit_rating
B    4119
A    2503
C    1420
D     588
E     134
F      28
Name: count, dtype: int64


In [28]:
copy_train_df.columns

Index(['Age', 'Income', 'Home_ownership', 'Employment_length', 'Loan_intent',
       'internal_credit_rating', 'Loan_amount', 'Interest_rate',
       'Loan_percent_income', 'Default', 'Credit_history_length',
       'Loan_status', 'is_eligible', 'age_employment_ratio',
       'income_per_credit_year', 'Age_Group', 'Income_Group',
       'Employment_Length_Group', 'Loan_Amount_Group', 'Interest_Rate_Group',
       'Loan_Percent_Income_Group', 'Credit_History_Group',
       'age_employment_ratio_Group', 'income_per_credit_year_Group',
       'Age_Group_woe', 'Income_Group_woe', 'Home_ownership_woe',
       'Employment_Length_Group_woe', 'Loan_intent_woe',
       'internal_credit_rating_woe', 'Loan_Amount_Group_woe',
       'Interest_Rate_Group_woe', 'Loan_Percent_Income_Group_woe',
       'Credit_History_Group_woe', 'age_employment_ratio_Group_woe',
       'income_per_credit_year_Group_woe', 'prob', 'credit_score'],
      dtype='object')

In [29]:
# 
# High Risk Combo
copy_train_df['high_risk_flag'] = (
    (copy_train_df['Loan_percent_income'] > 0.4) & 
    (copy_train_df['credit_score'] < 600) 
).astype(int)

In [30]:
# 
# High Risk Combo
copy_test_df['high_risk_flag'] = (
    (copy_test_df['Loan_percent_income'] > 0.4) & 
    (copy_test_df['credit_score'] < 600)
).astype(int)

In [31]:
copy_train_df.columns, copy_test_df.columns

(Index(['Age', 'Income', 'Home_ownership', 'Employment_length', 'Loan_intent',
        'internal_credit_rating', 'Loan_amount', 'Interest_rate',
        'Loan_percent_income', 'Default', 'Credit_history_length',
        'Loan_status', 'is_eligible', 'age_employment_ratio',
        'income_per_credit_year', 'Age_Group', 'Income_Group',
        'Employment_Length_Group', 'Loan_Amount_Group', 'Interest_Rate_Group',
        'Loan_Percent_Income_Group', 'Credit_History_Group',
        'age_employment_ratio_Group', 'income_per_credit_year_Group',
        'Age_Group_woe', 'Income_Group_woe', 'Home_ownership_woe',
        'Employment_Length_Group_woe', 'Loan_intent_woe',
        'internal_credit_rating_woe', 'Loan_Amount_Group_woe',
        'Interest_Rate_Group_woe', 'Loan_Percent_Income_Group_woe',
        'Credit_History_Group_woe', 'age_employment_ratio_Group_woe',
        'income_per_credit_year_Group_woe', 'prob', 'credit_score',
        'high_risk_flag'],
       dtype='object'),
 Index([

Here age is weak predictor so we dont need add that in it so we remove it

In [32]:
bins = [0, 18, 25, 35, 45, 60, 100]
labels = ['<18', '18-25', '26-35', '36-45', '46-60', '60+']

copy_train_df['age_group'] = pd.cut(copy_train_df['Age'], bins=bins, labels=labels)
copy_test_df['age_group'] = pd.cut(copy_test_df['Age'], bins=bins, labels=labels)

In [33]:
copy_train_df.drop(columns=['Age'], inplace=True)
copy_test_df.drop(columns=['Age'], inplace=True)

#### Remove derived WOE and grouped features, keep only original columns

In [34]:
# remove derived WOE and grouped features, keep only original columns
copy_train_df = copy_train_df[copy_train_df.columns[~copy_train_df.columns.str.contains('_Group|_woe|prob')]]

In [35]:
# remove derived WOE and grouped features, keep only original columns
copy_test_df = copy_test_df[copy_test_df.columns[~copy_test_df.columns.str.contains('_Group|_woe|prob')]]

In [36]:
train_df = copy_train_df.copy()
test_df = copy_test_df.copy()

In [37]:
# Save (industry common formats)
train_df.to_csv("../data/processed/train_df_transformed.csv", index=False)
test_df.to_csv("../data/processed/test_df_transformed.csv", index=False)

In [38]:
# Save (industry common formats)
train_df.to_csv("../data/processed/train_df_final_scaled.csv", index=False)
test_df.to_csv("../data/processed/test_df_final_scaled.csv", index=False)

#### Types Of Features

Numerical Features

In [39]:
copy_train_df.dtypes

Income                     float64
Home_ownership              object
Employment_length          float64
Loan_intent                 object
internal_credit_rating      object
Loan_amount                float64
Interest_rate              float64
Loan_percent_income        float64
Default                     object
Credit_history_length      float64
Loan_status                   bool
is_eligible                   bool
age_employment_ratio       float64
income_per_credit_year     float64
credit_score               float64
high_risk_flag               int64
age_group                 category
dtype: object

In [40]:
copy_train_df['Default'] = copy_train_df['Default'].map({'Y' : True, 'N' : False})
copy_train_df['age_group'] = copy_train_df['age_group'].astype('object')

copy_test_df['Default'] = copy_test_df['Default'].map({'Y' : True, 'N' : False})
copy_test_df['age_group'] = copy_test_df['age_group'].astype('object')

C:\Users\dell\AppData\Local\Temp\ipykernel_868\3110950753.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  copy_test_df['Default'] = copy_test_df['Default'].map({'Y' : True, 'N' : False})
C:\Users\dell\AppData\Local\Temp\ipykernel_868\3110950753.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  copy_test_df['age_group'] = copy_test_df['age_group'].astype('object')


In [41]:
# select all numerical features
train_num_features = copy_train_df.select_dtypes(include=['number']).columns.tolist()

# print total number of numerical features
print('Num of Numerical Features :', len(train_num_features))

# for test_df
test_num_features = train_num_features

Num of Numerical Features : 10


Categorical Features

In [42]:
# select all categorical features
train_cat_features = copy_train_df.select_dtypes(include=['object', 'bool']).columns.tolist()

# print total number of categorical features
print('Num of Categorical Features :', len(train_cat_features))

# for test df
test_cat_features = train_cat_features

Num of Categorical Features : 7


Bool Features

In [43]:
# identify discrete numerical features (low unique values)
train_bool_features = [feature for feature in train_cat_features if len(copy_train_df[feature].unique()) <= 2]

# print total number of discrete features
print('Num of Discrete Features :', len(train_bool_features))

# for test df
test_discrete_features = train_bool_features

Num of Discrete Features : 3


In [44]:
train_num_features, train_cat_features, train_bool_features

(['Income',
  'Employment_length',
  'Loan_amount',
  'Interest_rate',
  'Loan_percent_income',
  'Credit_history_length',
  'age_employment_ratio',
  'income_per_credit_year',
  'credit_score',
  'high_risk_flag'],
 ['Home_ownership',
  'Loan_intent',
  'internal_credit_rating',
  'Default',
  'Loan_status',
  'is_eligible',
  'age_group'],
 ['Default', 'Loan_status', 'is_eligible'])

#### Outlier features

In [45]:
# get all numeric features from dataframe
train_numeric_features = copy_train_df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# list of features identified with outliers
train_outlier_features = [
    "Income",
    "Employment_length",
    "Loan_amount",
    "Loan_percent_income",
    "Credit_history_length",
    "income_per_credit_year",
    "credit_score",
]

test_outlier_features = train_outlier_features

# remove outlier features from numeric feature list
train_numeric__without_outlier_features = [x for x in train_numeric_features if x not in train_outlier_features]

test_numeric__without_outlier_features = train_numeric__without_outlier_features

In [46]:
rating_map = {'A':7, 'B':6, 'C':5, 'D':4, 'E':3, 'F':2, 'G':1}

copy_train_df['internal_credit_rating'] = copy_train_df['internal_credit_rating'].map(rating_map)

copy_test_df['internal_credit_rating'] = copy_test_df['internal_credit_rating'].map(rating_map)

C:\Users\dell\AppData\Local\Temp\ipykernel_868\3476917139.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  copy_test_df['internal_credit_rating'] = copy_test_df['internal_credit_rating'].map(rating_map)


In [47]:
train_num_features, train_numeric__without_outlier_features

(['Income',
  'Employment_length',
  'Loan_amount',
  'Interest_rate',
  'Loan_percent_income',
  'Credit_history_length',
  'age_employment_ratio',
  'income_per_credit_year',
  'credit_score',
  'high_risk_flag'],
 ['Interest_rate', 'age_employment_ratio', 'high_risk_flag'])

In [48]:
# for scaling the numerical values of internal credit rating column + interest rate
train_numeric__without_outlier_features = train_numeric__without_outlier_features + ['internal_credit_rating']

# for test df
test_numeric__without_outlier_features = train_numeric__without_outlier_features

#### Split X and y

In [49]:
# split dataset into features (X) and target (y)
X = copy_train_df.drop(['Loan_status','age_employment_ratio','income_per_credit_year','credit_score'], axis=1)
y = copy_train_df['Loan_status']

In [50]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=32)

#### Preprocessig the columns

In [51]:
# X_train = pd.read_csv('X_train.csv')
# X_test = pd.read_csv('X_test.csv')
# y_test = pd.read_csv('y_test.csv')
# y_train = pd.read_csv('y_train.csv')

In [52]:
# train_df = pd.read_csv('train_df.csv')

# copy_train_df = train_df
# copy_test_df = pd.read_csv('test_df.csv')

In [53]:
# import preprocessing tools
from sklearn.preprocessing import StandardScaler, PowerTransformer, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# copy dataset
dataC = copy_train_df.copy()

# pipeline for normal numeric features (scaling)
numeric_pipeline = Pipeline(steps=[
    ("StandardScaler", StandardScaler())
])

# pipeline for outlier-prone features (power transform)
outlier_features_pipeline = Pipeline(steps=[
    ("Transformers", PowerTransformer(standardize=True))
])

# pipeline for categorical features (one-hot encoding)
cat_pipeline_onehot_encoding = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# combine all pipelines into one preprocessor
preprocessor = ColumnTransformer(
    [
        ("numeric_pipeline", numeric_pipeline, 
         [col for col in train_numeric__without_outlier_features if not (col.startswith('age_employ'))]),
        ("outlier_feature_pipeline", outlier_features_pipeline,
         [col for col in train_outlier_features if not (col.startswith('income_per') or col.startswith('credit_sco'))]),
        ("encoding_feature_pipeline", cat_pipeline_onehot_encoding,
         [col for col in train_cat_features if not (col.startswith('internal') or col.startswith('Loan_stat'))])
    ]
)

# fit and transform training data
X_train_prep = preprocessor.fit_transform(X_train)
X_train_prep_df = pd.DataFrame(X_train_prep, columns=preprocessor.get_feature_names_out())

# transform test data
X_test_prep = preprocessor.transform(X_test)
X_test_prep_df = pd.DataFrame(X_test_prep, columns=preprocessor.get_feature_names_out())

# transform on test data
copy_test_df = preprocessor.transform(copy_test_df)
copy_test_df = pd.DataFrame(copy_test_df, columns=preprocessor.get_feature_names_out())

In [54]:
train_numeric__without_outlier_features, train_outlier_features, train_cat_features

(['Interest_rate',
  'age_employment_ratio',
  'high_risk_flag',
  'internal_credit_rating'],
 ['Income',
  'Employment_length',
  'Loan_amount',
  'Loan_percent_income',
  'Credit_history_length',
  'income_per_credit_year',
  'credit_score'],
 ['Home_ownership',
  'Loan_intent',
  'internal_credit_rating',
  'Default',
  'Loan_status',
  'is_eligible',
  'age_group'])

#### Renaming the columns

In [55]:
# get feature names after preprocessing
columns = preprocessor.get_feature_names_out()

# convert transformed array into dataframe
scaled_data_X_train = pd.DataFrame(X_train_prep, columns=columns)
scaled_data__copy_test_df = pd.DataFrame(copy_test_df, columns=columns)
scaled_data_X_test = pd.DataFrame(X_test_prep, columns=columns)

# display message
print("After scaling, let's have a glimpse of the scaled dataset :")

# # clean column names (remove pipeline prefixes)
scaled_data_X_train.columns = scaled_data_X_train.columns.str.replace("numeric_pipeline__", "")
scaled_data_X_train.columns = scaled_data_X_train.columns.str.replace("outlier_feature_pipeline__", "")
scaled_data_X_train.columns = scaled_data_X_train.columns.str.replace("encoding_feature_pipeline__", "")

scaled_data__copy_test_df.columns = scaled_data__copy_test_df.columns.str.replace("numeric_pipeline__", "")
scaled_data__copy_test_df.columns = scaled_data__copy_test_df.columns.str.replace("outlier_feature_pipeline__", "")
scaled_data__copy_test_df.columns = scaled_data__copy_test_df.columns.str.replace("encoding_feature_pipeline__", "")

# show first rows
scaled_data_X_train.head()
scaled_data__copy_test_df.head()
scaled_data_X_test.columns = scaled_data_X_test.columns.str.replace("numeric_pipeline__", "")
scaled_data_X_test.columns = scaled_data_X_test.columns.str.replace("outlier_feature_pipeline__", "")
scaled_data_X_test.columns = scaled_data_X_test.columns.str.replace("encoding_feature_pipeline__", "")


After scaling, let's have a glimpse of the scaled dataset :


In [56]:
scaled_data__copy_test_df.columns, scaled_data_X_train.columns

(Index(['Interest_rate', 'high_risk_flag', 'internal_credit_rating', 'Income',
        'Employment_length', 'Loan_amount', 'Loan_percent_income',
        'Credit_history_length', 'Home_ownership_Mortgage',
        'Home_ownership_Other', 'Home_ownership_Own', 'Home_ownership_Rent',
        'Loan_intent_Debtconsolidation', 'Loan_intent_Education',
        'Loan_intent_Homeimprovement', 'Loan_intent_Medical',
        'Loan_intent_Personal', 'Loan_intent_Venture', 'Default_False',
        'Default_True', 'is_eligible_False', 'is_eligible_True',
        'age_group_18-25', 'age_group_26-35', 'age_group_36-45',
        'age_group_46-60', 'age_group_60+'],
       dtype='object'),
 Index(['Interest_rate', 'high_risk_flag', 'internal_credit_rating', 'Income',
        'Employment_length', 'Loan_amount', 'Loan_percent_income',
        'Credit_history_length', 'Home_ownership_Mortgage',
        'Home_ownership_Other', 'Home_ownership_Own', 'Home_ownership_Rent',
        'Loan_intent_Debtconsolidat

In [57]:
np.savez('x_y_splits.npz', 
                X_train_prep=X_train_prep,
                X_test_prep=X_test_prep,
                y_train=y_train,
                y_test=y_test)

In [58]:
# Save (industry common formats)
scaled_data_X_train.to_csv("../data/processed/scaled_data_X_train.csv", index=False)
scaled_data__copy_test_df.to_csv("../data/processed/scaled_data__copy_test_df.csv", index=False)
scaled_data_X_test.to_csv("../data/processed/scaled_data_X_test.csv", index=False)